# DAPO算法: 理论与代码逐块对应

实现状态：教学骨架；未运行论文训练复现。

本 Notebook 将 DAPO 的四项关键技术与最小公式实现逐块对应。

In [ ]:
import torch
torch.manual_seed(42)

---

## 1. Clip-Higher (解耦裁剪)

### 公式
$$\text{clip}^{DAPO}(r, A) = \begin{cases}
\min(r, 1+\epsilon_{high}) & A > 0 \text{ (只裁上界)}\\
\max(r, 1-\epsilon_{low}) & A < 0 \text{ (只裁下界)}
\end{cases}$$

DAPO使用 $\epsilon_{high}=0.28 > \epsilon_{low}=0.2$，鼓励探索. 

In [ ]:
def dapo_clip(ratio, advantages, eps_high=0.28, eps_low=0.2):
    """
    DAPO解耦裁剪
    
    A > 0: 只裁上界 (允许更多探索)
    A < 0: 只裁下界 (防止过度惩罚)
    """
    clipped = torch.where(
        advantages > 0,
        torch.clamp(ratio, max=1 + eps_high),
        torch.clamp(ratio, min=1 - eps_low)
    )
    return clipped

# 演示
ratio = torch.tensor([0.7, 0.9, 1.1, 1.5])
advantages = torch.tensor([1.0, 1.0, -1.0, -1.0])

clipped = dapo_clip(ratio, advantages)
print(f"概率比 r:  {ratio.tolist()}")
print(f"优势 A:    {advantages.tolist()}")
print(f"裁剪后:    {clipped.tolist()}")
print()
print("解读:")
print("  - r=0.7, A>0: 不裁剪 (下界不影响正优势)")
print("  - r=1.5, A<0: 不裁剪 (上界不影响负优势)")

---

## 2. Dynamic Sampling (动态采样)

排除组内奖励完全相同、无法形成相对优势的 prompts；继续补采样，直到得到固定数量的有效 prompts。该条件不依赖奖励编码。

In [ ]:
def filter_informative_prompts(rewards, group_size):
    """保留组内奖励不全相同的 prompts。"""
    rewards_grouped = rewards.view(-1, group_size)
    return rewards_grouped.amax(dim=1) > rewards_grouped.amin(dim=1)

# 演示: 3个prompts，每个4个responses
rewards = torch.tensor([
    1.0, 0.0, 0.5, 0.0,  # Prompt 1: 有非零 ✓
    0.0, 0.0, 0.0, 0.0,  # Prompt 2: 全零 ✗
    0.0, 1.0, 0.0, 1.0   # Prompt 3: 有非零 ✓
])

valid = filter_informative_prompts(rewards, group_size=4)
print(f"奖励: {rewards.tolist()}")
print(f"有效prompts: {valid.tolist()}")
print("\n结果: Prompt 2被过滤，避免无效更新")

---

## 3. Token-Level Loss

### 公式
$$L^{DAPO} = -\frac{1}{\sum_i |y_i|}\sum_i\sum_t \min\left(r_{i,t}A_i,\operatorname{clip}(r_{i,t})A_i\right)$$

比率仍逐 token 计算；DAPO 的关键变化是把整个有效批的 token 放到同一个分母中，而不是先对每条回复分别除以自身长度。

In [ ]:
def token_level_loss(per_token_ratio, advantages, mask, eps_low=0.2, eps_high=0.28):
    """
    Token级损失
    
    L = -Σ_iΣ_t min(rA, clip(r)A) / Σ_i|y_i|
    """
    # 扩展优势到token维度
    adv_expanded = advantages.unsqueeze(-1)  # [B, 1]
    
    clipped = torch.where(
        adv_expanded > 0,
        torch.clamp(per_token_ratio, max=1 + eps_high),
        torch.clamp(per_token_ratio, min=1 - eps_low),
    )
    per_token_loss = -torch.minimum(
        per_token_ratio * adv_expanded, clipped * adv_expanded
    )
    
    # 掩码求和
    loss = (per_token_loss * mask).sum() / mask.sum().clamp_min(1)
    return loss

# 模拟
B, T = 2, 5
per_token_ratio = torch.ones(B, T)  # [2, 5]
advantages = torch.tensor([0.5, -0.5])  # [2]
mask = torch.ones(B, T)

loss = token_level_loss(per_token_ratio, advantages, mask)
print(f"Token级损失: {loss.item():.4f}")

---

## 4. Overlong Reward Shaping

### 公式
令 $L_{start}=L_{max}-L_{cache}$。DAPO Eq. (13) 在 $[L_{start},L_{max}]$ 内把长度奖励从 0 线性降到 -1；超过 $L_{max}$ 时保持 -1。论文配方使用 $L_{max}=16384$、$L_{cache}=4096$。

In [ ]:
def overlong_shaping(rewards, lengths, max_len=16384, cache_len=4096):
    """DAPO Eq. (13) 的分段线性长度奖励。"""
    if cache_len <= 0 or cache_len > max_len:
        raise ValueError("cache_len 必须位于 (0, max_len] 内")
    start = max_len - cache_len
    lengths = lengths.to(rewards.dtype)
    soft = (start - lengths) / cache_len
    length_reward = torch.where(
        lengths <= start, torch.zeros_like(lengths),
        torch.where(lengths <= max_len, soft, -torch.ones_like(lengths)),
    )
    return rewards + length_reward

# 演示
rewards = torch.zeros(4)
lengths = torch.tensor([12000, 14000, 16384, 17000])

shaped = overlong_shaping(rewards, lengths)
print(f"原始奖励: {rewards.tolist()}")
print(f"回复长度: {lengths.tolist()}")
print(f"塑造后:   {shaped.tolist()}")
print("\n结果: 长度600超过500，被惩罚")

---

## 5. 总结

| 技术 | 解决的问题 | 核心改进 |
|------|------------|----------|
| Clip-Higher | 熵坍缩 | 解耦裁剪，$\epsilon_{high} > \epsilon_{low}$ |
| Dynamic Sampling | 无相对优势的组 | 排除组内奖励全同并补足固定有效批 |
| Token-Level Loss | 回复长度归约偏差 | 逐 token 比率、全局有效 token 分母 |
| Overlong Shaping | 截断惩罚突变 | 最大长度前窗口内线性软惩罚 |